<a href="https://colab.research.google.com/github/Poojarautela03/ABTALKS/blob/main/day-14-faiss-semantic-search/semantic-search-faiss.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [30]:
!pip install -q google-generativeai faiss-cpu numpy

import numpy as np
import faiss
import google.generativeai as genai
import os
from google.colab import userdata

genai.configure(api_key=userdata.get('GEMINI_API_KEY'))
print("Setup done.")

Setup done.


In [31]:
corpus = [
    "NASA's Artemis program aims to return humans to the Moon by the mid-2020s.",
    "The James Webb Space Telescope captures infrared images of distant galaxies.",
    "SpaceX's Starship is designed to be a fully reusable launch vehicle.",
    "Mars rovers like Perseverance search for signs of ancient microbial life.",
    "The International Space Station orbits Earth roughly every 90 minutes.",
    "Astronauts experience microgravity, which affects muscle and bone density.",
    "The Hubble Space Telescope has operated in orbit since 1990.",
    "Rocket propulsion relies on Newton's third law of motion.",
    "Solar panels power most spacecraft by converting sunlight into electricity.",
    "The Apollo 11 mission landed the first humans on the Moon in 1969.",
    "Black holes have gravitational fields so strong that not even light escapes.",
    "The Voyager probes are now traveling through interstellar space.",
    "Satellite constellations like Starlink provide global internet coverage.",
    "Exoplanets are planets that orbit stars outside our solar system.",
    "Space agencies use spectroscopy to determine the composition of distant stars.",
    "The chef prepared a delicious pasta dish for the dinner party.",
    "She adopted a golden retriever puppy from the local shelter.",
    "The stock market experienced significant volatility this quarter.",
    "He practiced piano for two hours every evening after school.",
    "The bakery down the street sells fresh croissants every morning.",
    "Machine learning models require large amounts of labeled training data.",
    "Neural networks are loosely inspired by the structure of the human brain.",
    "Cloud computing lets businesses scale infrastructure on demand.",
    "Data preprocessing is a crucial step before training any ML model.",
    "GPUs significantly speed up deep learning training compared to CPUs.",
    "Natural language processing helps computers understand human text.",
    "Reinforcement learning trains agents through rewards and penalties.",
    "Computer vision enables machines to interpret and analyze images.",
    "APIs allow different software systems to communicate with each other.",
    "Cybersecurity protects systems and data from unauthorized access.",
    "The mountain hike offered a breathtaking view at sunset.",
    "The garden was full of blooming roses in early spring.",
    "They watched a thrilling action movie at the cinema last night.",
    "The football match ended in a dramatic penalty shootout.",
    "She baked chocolate chip cookies for the neighborhood party.",
    "The airplane landed safely despite the heavy storm.",
    "We booked flights for our summer vacation to Europe.",
    "The airport was extremely crowded during the holiday season.",
    "Traveling by train offers scenic views of the countryside.",
    "The restaurant received excellent reviews for its friendly service.",
    "Renewable energy sources include solar, wind, and hydroelectric power.",
    "Electric vehicles are becoming more affordable and widely available.",
    "Climate change is causing more frequent extreme weather events.",
    "Recycling programs help reduce the amount of landfill waste.",
    "Deforestation contributes significantly to global carbon emissions.",
    "The novel's plot twist surprised readers in the final chapter.",
    "Poetry often uses metaphor to convey complex emotions concisely.",
    "The museum exhibit featured paintings from the Renaissance period.",
    "Classical music concerts often feature works by Beethoven and Mozart.",
    "The library extended its hours during final exam week.",
    "Yoga and meditation are popular practices for reducing stress.",
    "A balanced diet includes proteins, carbohydrates, and healthy fats."
]

print(f"Corpus size: {len(corpus)} documents")

Corpus size: 52 documents


In [32]:
def get_embeddings(texts, model="models/gemini-embedding-001"):
    embeddings = []
    for text in texts:
        result = genai.embed_content(model=model, content=text)
        embeddings.append(result['embedding'])
    return embeddings

corpus_embeddings = get_embeddings(corpus)
corpus_vectors = np.array(corpus_embeddings, dtype=np.float32)

print(f"Embedding matrix shape: {corpus_vectors.shape}")

Embedding matrix shape: (52, 3072)


In [33]:
dimension = corpus_vectors.shape[1]  # 1536 for text-embedding-3-small
index = faiss.IndexFlatL2(dimension)  # flat index, exact L2 distance search
index.add(corpus_vectors)

print(f"FAISS index size: {index.ntotal} vectors")  # should print 50
assert index.ntotal == len(corpus), "Mismatch between corpus and index size!"

FAISS index size: 52 vectors


In [34]:
def semantic_search(query, top_k=3):
    query_embedding = get_embeddings([query])[0]
    query_vector = np.array([query_embedding], dtype=np.float32)

    distances, indices = index.search(query_vector, top_k)

    results = []
    for dist, idx in zip(distances[0], indices[0]):
        results.append((corpus[idx], dist))
    return results

In [35]:
def keyword_search(query, corpus, top_k=3):
    query_words = set(query.lower().split())

    scores = []
    for doc in corpus:
        doc_words = set(doc.lower().replace('.', '').split())
        overlap = len(query_words & doc_words)
        scores.append((doc, overlap))

    scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:top_k]

In [36]:
test_queries = [
    "AI in healthcare",                      # semantic should win
    "training a computer to recognize images", # semantic should win
    "vacation travel plans",                  # semantic should win
    "renewable power sources",                # keyword may tie/win
    "Moon landing",                           # both should do well
    "electric cars",                          # keyword may win (exact words)
    "reducing environmental impact",          # semantic should win
    "space telescope",                        # both should do well
    "pizza recipe",                           # out of domain — both should fail gracefully
    "friendly restaurant service"             # keyword may win
]

for q in test_queries:
    print("=" * 70)
    print(f"QUERY: '{q}'\n")

    print("Semantic search (FAISS):")
    for doc, dist in semantic_search(q, top_k=3):
        print(f"  [dist={dist:.4f}] {doc}")

    print("\nKeyword search (word overlap):")
    for doc, overlap in keyword_search(q, corpus, top_k=3):
        print(f"  [overlap={overlap}] {doc}")
    print()

QUERY: 'AI in healthcare'

Semantic search (FAISS):
  [dist=0.7785] Natural language processing helps computers understand human text.
  [dist=0.7895] Computer vision enables machines to interpret and analyze images.
  [dist=0.8038] Data preprocessing is a crucial step before training any ML model.

Keyword search (word overlap):
  [overlap=1] The Hubble Space Telescope has operated in orbit since 1990.
  [overlap=1] The Apollo 11 mission landed the first humans on the Moon in 1969.
  [overlap=1] The garden was full of blooming roses in early spring.

QUERY: 'training a computer to recognize images'

Semantic search (FAISS):
  [dist=0.5719] Computer vision enables machines to interpret and analyze images.
  [dist=0.6763] Machine learning models require large amounts of labeled training data.
  [dist=0.7127] GPUs significantly speed up deep learning training compared to CPUs.

Keyword search (word overlap):
  [overlap=3] Computer vision enables machines to interpret and analyze images.
